In [ ]:
!pip install pygwalker

In [ ]:
!pip install bokeh

In [ ]:
# IMPORTAR LIBRERÍAS

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource
from math import pi
from bokeh.transform import cumsum
from bokeh.palettes import Category10
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.transform import factor_cmap

import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import pygwalker as pyg


In [ ]:
# Conectar con la cuenta de Google Drive con el entorno de Python

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Carga el dataset csv

df_csv = pd.read_csv('/content/drive/MyDrive/data/Teen_Mental_Health_Dataset.csv')

df_csv.head()

In [ ]:
# Información del dataset csv

print(df_csv.info())

print(df_csv.describe())

print(df_csv.isnull().sum())

In [ ]:
#Limpieza de csv

df_csv_limpio = df_csv.dropna()

print("Filas originales:", df_csv.shape)
print("Filas limpias:", df_csv_limpio.shape)

In [ ]:
# Conexión con la base de datos

conexion = sqlite3.connect('/content/drive/MyDrive/data/mental_health.sqlite')

cursor = conexion.cursor()

# Consultas SQL para obtener tablas de SQLite

query = """
SELECT *
FROM Answer
"""

# cargar datos en pandas
df_sql = pd.read_sql_query(query, conexion)

# mostrar datos
df_sql.head()

In [ ]:
# Asignación de identificador de encuesta (SurveyID)

df_csv['SurveyID'] = 2019

In [ ]:
# Realiza una unión (merge) entre dos DataFrames de pandas: df_csv y df_sql, utilizando la columna SurveyID como clave en común.

df_unido = pd.merge(
    df_csv,
    df_sql,
    on='SurveyID',
    how='inner'
)

df_unido.head()

In [ ]:
# print(df_unido.info())

# print(df_unido.shape)

In [ ]:
# Consultas SQL para obtener tablas de SQLite

query = """
SELECT *
FROM Survey
"""

# cargar datos en pandas
df_sql = pd.read_sql_query(query, conexion)

# mostrar datos
df_sql.head()

In [ ]:
# Consultas SQL para obtener tablas de SQLite

query = """
SELECT SurveyID, COUNT(*) as total_respuestas
FROM Answer
GROUP BY SurveyID
"""

df_respuestas = pd.read_sql_query(query, conexion)

print(df_respuestas)

In [ ]:
# Consultas SQL para obtener tablas de SQLite

query = """
SELECT SurveyID, COUNT(*) as total
FROM Answer
GROUP BY SurveyID
"""

df_pie = pd.read_sql_query(query, conexion)

print(df_pie)

In [ ]:
#Información general del dataset SQLite

print(df_sql.info())

print(df_sql.describe())

print(df_sql.isnull().sum())

In [ ]:
# Nombres o encabezados de todas las columnas del DataFrame df_unido

print(df_unido.columns.tolist())

In [ ]:
# Gráfico de barras de la cantidad de respuestas por encuesta

plt.figure(figsize=(8,5))

plt.bar(
    df_respuestas['SurveyID'].astype(str),
    df_respuestas['total_respuestas']
)

plt.title('Cantidad de respuestas por encuesta')
plt.xlabel('SurveyID')
plt.ylabel('Total de respuestas')

plt.show()

In [ ]:
# Gráfico circular del porcentaje de respuesta por encuesta

plt.figure(figsize=(7,7))

plt.pie(
    df_pie['total'],
    labels=df_pie['SurveyID'],
    autopct='%1.1f%%'
)

plt.title('Porcentaje de respuestas por encuesta')

plt.show()

In [ ]:
# Gráfico de Histograma de la distribución de edades

plt.figure(figsize=(8,5))

plt.hist(
    df_csv_limpio['age'],
    bins=10
)

plt.title('Distribución de edades')
plt.xlabel('Edad')
plt.ylabel('Frecuencia')

plt.show()

In [ ]:
# Gráfico de cantidad por género

genero = df_csv_limpio['gender'].value_counts()

plt.figure(figsize=(6,4))

plt.bar(
    genero.index,
    genero.values
)

plt.title('Cantidad de estudiantes por género')
plt.xlabel('Género')
plt.ylabel('Cantidad')

plt.show()

In [ ]:
# Gráfico de nivel de estrés

estres = df_csv_limpio['stress_level'].value_counts()

plt.figure(figsize=(6,6))

plt.pie(
    estres.values,
    labels=estres.index,
    autopct='%1.1f%%'
)

plt.title('Distribución del nivel de estrés')

plt.show()

In [ ]:
# activar bokeh en colab
output_notebook()

# contar niveles de estrés
estres = df_csv_limpio['stress_level'].value_counts()

# crear dataframe
df_estres = pd.DataFrame({
    'nivel': estres.index.astype(str),
    'cantidad': estres.values
})

source = ColumnDataSource(df_estres)

# crear gráfico
p = figure(
    x_range=df_estres['nivel'],
    height=400,
    width=700,
    title='Cantidad de estudiantes por nivel de estrés',
    toolbar_location=None,
    tools=''
)

p.vbar(
    x='nivel',
    top='cantidad',
    width=0.5,
    source=source
)

p.xaxis.axis_label = 'Nivel de estrés'
p.yaxis.axis_label = 'Cantidad'

show(p)

In [ ]:
# Se crean el tamaño de las burbujas y transforma los niveles de estrés en valores numéricos para poder utilizarlos como tamaños de burbujas

df_csv_limpio['bubble_size'] = (
    df_csv_limpio['stress_level']
    .astype('category')
    .cat.codes + 1
) * 8

In [ ]:
# Convierte el DataFrame de Pandas en una fuente de datos que Bokeh puede utilizar.

source = ColumnDataSource(df_csv_limpio)

In [ ]:
# Crear una figura interactiva de Bokeh
p = figure(

    # Define el ancho del gráfico en píxeles
    width=850,

    # Define la altura del gráfico en píxeles
    height=500,

    # Título principal del gráfico
    title='Relación entre edad y nivel de estrés',

    # Herramientas interactivas habilitadas:
    # pan -> mover gráfico
    # wheel_zoom -> zoom con rueda del mouse
    # box_zoom -> zoom seleccionando área
    # reset -> restaurar gráfico
    # save -> guardar imagen
    # hover -> mostrar información al pasar el mouse
    tools='pan,wheel_zoom,box_zoom,reset,save,hover',

    # Información emergente que aparece al pasar el cursor
    tooltips=[
        ('Edad', '@age'),
        ('Estrés', '@stress_level'),
        ('Género', '@gender')
    ]
)

# Crear gráfico tipo burbuja (scatter plot)
p.scatter(

    # Variable del eje X
    x='age',

    # Variable del eje Y
    y='bubble_size',

    # Tamaño de las burbujas
    size='bubble_size',

    # Fuente de datos usada por Bokeh
    source=source,

    # Transparencia de las burbujas
    fill_alpha=0.6,

    # Colorear automáticamente según el nivel de estrés
    color=factor_cmap(

        # Columna usada para asignar colores
        'stress_level',

        # Paleta de colores
        palette=Category10[10],

        # Categorías únicas de niveles de estrés
        factors=list(
            df_csv_limpio['stress_level']
            .astype(str)
            .unique()
        )
    ),

    # Crear leyenda usando la categoría stress_level
    legend_field='stress_level'
)

# Nombre del eje X
p.xaxis.axis_label = 'Edad'

# Nombre del eje Y
p.yaxis.axis_label = 'Nivel de estrés'

# Ubicación de la leyenda
p.legend.location = 'top_left'

# Mostrar gráfico interactivo
show(p)

In [ ]:
# pyg.walk(df_csv)

In [ ]:
# Guardar

df_csv_limpio.to_csv(
    'dataset_final.csv',
    index=False
)

print("Dataset final exportado correctamente")

In [ ]:
# Exportar

with open('database.sql', 'w') as f:
    for line in conexion.iterdump():
        f.write('%s\n' % line)

print("Archivo SQL exportado")

In [ ]:
#Descargar archivos finales

from google.colab import files

files.download('dataset_final.csv')

files.download('database.sql')